# Pydantic

它通过在运行时强制执行类型提示，确保数据的正确性和一致性，是 生产场景首选 。
## 基本使用
前提：
- 所有结构化输出的数据模型必须继承`BaseModel`
- 使用`类型提示`
- 使用`Field()`添加字段默认值和描述

In [1]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool  # 从 core 导入，避免 langgraph 版本冲突
from rich import print as rprint
from langchain_qwq import ChatQwen
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)
model = ChatQwen(
    model="deepseek-v4-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

In [2]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    """
    人物信息
    """
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    job: str = Field(description="职业")

model_with_structured_output = model.with_structured_output(Person)

result = model_with_structured_output.invoke('张三是一名30岁的前端开发工程师')
rprint(result) # Person(name='张三', age=30, job='前端开发工程师')
print(type(result)) # <class 'Person'>

Person(name='张三', age=30, job='前端开发工程师')

<class '__main__.Person'>


In [3]:

class Movie(BaseModel):
    """
    电影信息
    """
    title: str = Field(description="电影名称")
    year: int = Field(description="上映年份")
    rating: float = Field(description="评分")
    description: str = Field(description="电影描述")

model_with_structured_output = model.with_structured_output(Movie)
result = model_with_structured_output.invoke("给我一个电影的信息")
rprint(result)
rprint(result.model_dump()) # 将模型转换为字典

Movie(
    title='肖申克的救赎',
    year=1994,
    rating=9.7,
    description='银行家安迪因被误判杀害妻子及其情人而入狱，在肖申克监狱中结识了囚犯瑞德。经过多年隐忍与智慧谋划，安
迪最终成功越狱并揭露了监狱长的腐败，重获自由。'
)

{
    'title': '肖申克的救赎',
    'year': 1994,
    'rating': 9.7,
    'description': 
'银行家安迪因被误判杀害妻子及其情人而入狱，在肖申克监狱中结识了囚犯瑞德。经过多年隐忍与智慧谋划，安迪最终成功越狱并
揭露了监狱长的腐败，重获自由。'
}

### 可选字段
使用`Optional`修饰字段，表示该字段可以为空。

In [4]:
from typing import Optional

class Movie(BaseModel):
    title: str = Field(description="电影名称")
    year: int = Field(description="上映年份")
    rating: float = Field(description="评分")
    description: Optional[str] = Field(description="电影描述")

model_with_structured_output = model.with_structured_output(Movie)
result = model_with_structured_output.invoke("给我一个电影的信息，不要添加描述")
rprint(result.model_dump())


{'title': '肖申克的救赎', 'year': 1994, 'rating': 9.7, 'description': ''}

### 默认值
LLM 未提供的信息会使用默认值。格式如下：
- `Field(default="默认值", description="描述")`

注意：不同模型提供商对default字段的支持是不同的

In [5]:
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(1,description="年龄")
    occupation: str = Field(description="职业")
structured_llm = model.with_structured_output(Person)
result = structured_llm.invoke("张三是一名医生")
rprint(result)

Person(name='张三', age=1, occupation='医生')

### 枚举类型

In [6]:
from enum import Enum

class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"

class Task(BaseModel):
    title: str
    priority: Priority # 只能是 LOW/MEDIUM/HIGH

class Status(str, Enum):
    ACTIVE = "激活"
    INACTIVE = "未激活"

class User(BaseModel):
    status: Status # 只能是 ACTIVE 或 INACTIVE

structured_llm = model.with_structured_output(User)

class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")

structured_llm = model.with_structured_output(CustomerInfo)
result = structured_llm.invoke("客户姓名是张三，电话号码是1234567890，邮箱是zhangsan@example.com，问题描述是电池电量不足，紧急程度是高")
rprint(result.model_dump()['urgency'])

conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""

result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
rprint(result)

高

CustomerInfo(
    name='王小明',
    phone='138-1234-5678',
    email='null',
    issue='订单一直没发货',
    urgency=<Priority.HIGH: '高'>
)

### 列表提取

In [ ]:
from typing import List
class Person(BaseModel):
    """人物信息"""
    name: str
    age: int

class PersonList(BaseModel):
    """人物列表信息"""
    people: List[Person] # 多个 Person 对象
    
structured_llm = model.with_structured_output(PersonList)
result = structured_llm.invoke("张三 30岁，李四 25岁")
rprint(result)

people=[Person(name='张三', age=30), Person(name='李四', age=25)]


### 嵌套结构

In [ ]:
from pydantic import BaseModel
class Address(BaseModel):
    """地点描述"""
    city: str
    district: str
class Company(BaseModel):
    """公司信息"""
    name: str
    address: Address # 嵌套模型

structured_llm = model.with_structured_output(Company)

result = structured_llm.invoke("阿里巴巴在杭州滨江区")
rprint(result)

Company(name='阿里巴巴', address=Address(city='杭州', district='滨江区'))

### 限制条件

In [10]:
from pydantic import ValidationError

class User(BaseModel):
    name: str = Field(min_length=2, max_length=20)
    age: int = Field(ge=0, le=150)
    email: str

print("\n有效数据:")
try:
    user = User(name="张三", age=30, email="zhang@example.com")
    print(f"[OK] {user.name}, {user.age}, {user.email}")
except ValidationError as e:
    print(f"[FAIL] {e}")

print("\n无效数据（年龄超出范围）:")
try:
    user = User(name="李四", age=200, email="li@example.com")
    print(f"[OK] {user}")
except ValidationError as e:
    print(f"[FAIL] 验证失败（符合预期）: {e.errors()[0]['msg']}")


有效数据:
[OK] 张三, 30, zhang@example.com

无效数据（年龄超出范围）:
[FAIL] 验证失败（符合预期）: Input should be less than or equal to 150


In [ ]:
class Product(BaseModel):
    """产品信息（严格验证）"""
    name: str = Field(description="产品名称（字符串类型）", min_length=2)
    price: float = Field(description="价格，数字类型", gt=0)
    stock: int = Field(description="库存，整数类型", ge=0)
# 测试
structured_llm = model.with_structured_output(Product)
response = structured_llm.invoke("华为mate 80 promax 价格是7999，当前库存100")
# response = structured_llm.invoke("华为mate 80 promax 价格是-7999，当前库存-100") # 会报错，因为不符合限制条件
print(response)

name='华为mate 80 promax' price=7999.0 stock=100


## 工作流程图解

![LangChain Pydantic 结构化输出工作流程](../assets/pydantic-workflow.png)


1. **定义 Pydantic 模型** — 用 `BaseModel` 描述目标数据结构
2. **自动转换为 JSON Schema** — LangChain 调用 `model_json_schema()` 生成标准 Schema
3. **传递给 LLM** — Schema 作为工具/约束与用户提示词一起发送给模型
4. **生成结构化 JSON** — 模型按 Schema 输出 JSON 字符串
5. **Pydantic 验证与实例化** — LangChain 拦截输出，做类型检查、字段校验。LangChain 的 `PydanticStructuredOutputParser` （解析器）会接管工作。
   1. 解析（Parsing）： 将字符串解析为 Python 字典。
   2. 验证（Validation）： 将字典喂给你的 Pydantic 模型。Pydantic 会自动检查数据类型是否正确。如果模型漏掉了必填字段，或者类型错误，这里会直接抛出验证错误（或者触发 LangChain 的重试机制）。
1. **得到 Python 对象** — 返回可直接访问属性的 Pydantic 实例（如 `result.title`）